# Oracle AI Agent Memory — Developer Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/agent_memory/oracle_agent_memory_developer_guide.ipynb) [![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)


A hands-on, step-by-step guide to the `oracleagentmemory` AI Agent Package for AI developers. This notebook teaches you the API surface by building up an agent memory system from scratch — one concept per step.

**What you'll learn**
1. How to connect the package to an Oracle AI Database instance
2. The three core primitives: **users/agents**, **memories**, and **threads**
3. Manual vs. **automatic** (LLM-powered) memory extraction
4. How to retrieve relevant context with **vector search** and **context cards**
5. How to **scope** searches across multiple users and agents
6. How to plug the package into an LLM agent loop
7. How to **persist and resume** a conversation across sessions
8. How to build agents that adapt to and learn from new information — making them more reliable, believable, and capable over time

**Prerequisites**
- Python ≥ 3.10
- Docker running locally (for the Oracle AI Database container)
- An OpenAI API key with access to an embedding model (we use `text-embedding-3-small`)

## 1. Install the library

Install `oracleagentmemory` from PyPI. The `[litellm]` extra adds LiteLLM so you can call hosted embedding and chat models from any provider through a single interface.

In [ ]:
%pip install --upgrade pip
%pip install "oracleagentmemory[litellm]"

In [ ]:
import oracleagentmemory

## 2. Provide your OpenAI API key

The package uses an embedding model to turn text into vectors it can store and search. In this guide we use OpenAI's `text-embedding-3-small` via LiteLLM, which expects `OPENAI_API_KEY` in your environment.

The cell below uses `getpass` so the key is not written into the notebook.

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
print("OPENAI_API_KEY is set." if os.environ.get("OPENAI_API_KEY") else "Not set!")


## 3. Start an Oracle AI Database instance

The package stores everything (profiles, memories, messages, embeddings) in an Oracle AI Database. The easiest way to get one running locally is via Docker:

```bash
docker run -d \
  --name oracle-free \
  -p 1521:1521 \
  -e ORACLE_PASSWORD=VectorPwd_2025 \
  gvenzl/oracle-free:23-slim
```

Wait until `docker logs oracle-free` prints `DATABASE IS READY TO USE!`, then create a dedicated `VECTOR` schema:

```bash
docker exec oracle-free sqlplus sys/VectorPwd_2025@FREEPDB1 as sysdba <<'SQL'
CREATE USER VECTOR IDENTIFIED BY VectorPwd_2025;
GRANT CONNECT, RESOURCE, UNLIMITED TABLESPACE TO VECTOR;
SQL
```

If you already have a production Oracle instance, point `DB_CONNECT_STRING` at it instead and skip the Docker step.

## 4. Connect to Oracle

The package takes a raw `oracledb.Connection`. That's it — no custom connection pool, no ORM. You own the connection lifecycle, which means you can share it with other Oracle code in the same application.

In [ ]:
import oracledb

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print(f"Connected to Oracle {connection.version}")


## 5. Create a memory client

`OracleAgentMemory` is the top-level object you'll interact with. It owns the schema, the embedder, and (optionally) an LLM for automatic extraction.

Key constructor arguments:

| Argument | Purpose |
|---|---|
| `connection` | Your `oracledb` connection |
| `embedder` | A model identifier string (routed through LiteLLM) or an `Embedder` instance |
| `llm` | Optional — attaching one enables automatic memory extraction from threads |
| `extract_memories` | Whether to run extraction when messages are added to threads |
| `schema_policy` | `create_if_necessary` (safe default), `recreate` (drops & rebuilds), or `require_existing` |

For now we'll skip the LLM — we'll add one in step 9 to see automatic extraction.


In [ ]:
from oracleagentmemory.core import OracleAgentMemory

client = OracleAgentMemory(
    connection=connection,
    embedder="text-embedding-3-small",
    extract_memories=False,
    schema_policy="create_if_necessary",
)
print("Memory client ready.")


## 6. Register users and agents

Every memory in the package is **scoped** to a user, an agent, or both. Before you store memories, register the entities they belong to.

- `add_user(user_id, information)` — register a human user (the person the memories are *about*)
- `add_agent(agent_id, information)` — register an AI agent (the assistant *observing* and *creating* memories)

The `information` string is free-form profile text that can itself be searched.

In [ ]:
USER_ID = "dev-guide-user"
AGENT_ID = "dev-guide-agent"

for fn, eid, info in [
    (client.add_user,  USER_ID,  "Richmond — AI developer learning the AI Agent Package."),
    (client.add_agent, AGENT_ID, "Tutorial assistant for the AI Agent Package developer guide."),
]:
    try:
        fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError as exc:
        if "already exists" in str(exc):
            print(f"(already exists: {eid})")
        else:
            raise

## 7. Store your first memory

`add_memory(content, user_id=..., agent_id=..., thread_id=...)` saves a durable fact. The package embeds the content and stores both the text and its vector. Memories can be scoped to a user, an agent, a thread, or any combination.

In [ ]:
mem_id = client.add_memory(
    "Richmond is evaluating the AI Agent Package for a production RAG pipeline with 50k daily queries.",
    user_id=USER_ID,
    agent_id=AGENT_ID,
)
print(f"Stored memory: {mem_id}")

## 8. Search memories with vector retrieval

`search(query, user_id=..., agent_id=..., max_results=...)` runs a vector similarity search over stored memories. Results come back ranked by distance (lower = more similar).

Each result carries:

- `content` — the stored text
- `distance` — how close the query embedding was
- `record` — full database record, including `record_type` (`memory`, `message`, `user`, `agent`, etc.)
- `metadata`, `timestamp`, `id`


In [ ]:
results = client.search(
    "what is the user building?",
    user_id=USER_ID,
    max_results=5,
)
for r in results:
    print(f"[{r.record.record_type}] distance={r.distance:.3f}")
    print(f"  {r.content}")


## 9. Threads: the conversation primitive

A **thread** represents one conversation. Messages you add to a thread become searchable alongside memories, and — when you attach an LLM — they are automatically summarised and mined for new memories.

Create a thread scoped to a user and agent:


In [ ]:
thread = client.create_thread(user_id=USER_ID, agent_id=AGENT_ID)
print(f"Thread ID: {thread.thread_id}")

## 10. Add messages and read them back

Use `thread.add_messages([...])` with `Message(role=..., content=...)` objects. The roles follow the usual `system` / `user` / `assistant` convention.


In [ ]:
from oracleagentmemory.apis.thread import Message

thread.add_messages([
    Message(role="user",      content="I'm building a customer support agent for a SaaS product."),
    Message(role="assistant", content="Got it. Are you focused on tier-1 routing or full issue resolution?"),
    Message(role="user",      content="Full resolution — I want the agent to look up past tickets and draft replies."),
])

for m in thread.get_messages():
    print(f"[{m.role:10s}] {m.content}")

## 11. Retrieve a context card

`thread.get_context_card()` returns an XML-formatted block of the most relevant memories for the current state of the thread. It's the single most useful method for building prompts — instead of stuffing the whole history into every request, you pull a compact, query-relevant card.


In [ ]:
card = thread.get_context_card()
print(card or "(no memories retrieved yet)")


## 12. Compress the thread with `get_summary`

`thread.get_summary()` asks the LLM (if one is attached) for a compressed version of the conversation. Useful for long threads where you want to keep token counts low without losing continuity.

Without an LLM attached, it returns a plain concatenation — so let's add an LLM in the next step to see the real feature.


In [ ]:
summary = thread.get_summary()
print(summary[0].content if summary else "(no summary — attach an LLM next)")


## 13. Turn on automatic memory extraction

This is the headline feature. When you attach an `Llm` **and** pass `extract_memories=True`, the package will:

1. Watch every message you add to a thread
2. Periodically call the LLM to extract durable facts from recent messages
3. Store those facts as `memory` records, automatically scoped to the thread's user and agent
4. Maintain a rolling **context summary** for the thread

The relevant knobs on `create_thread()`:

| Argument | Meaning |
|---|---|
| `memory_extraction_frequency` | Run extraction every N messages |
| `memory_extraction_window` | How many recent messages to feed the extractor |
| `enable_context_summary` | Keep a running summary of the thread |
| `context_summary_update_frequency` | How often to refresh the summary |

In [ ]:
from oracleagentmemory.core.llms import Llm

llm = Llm("gpt-5.4")

smart_client = OracleAgentMemory(
    connection=connection,
    embedder="text-embedding-3-small",
    llm=llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
)

smart_thread = smart_client.create_thread(
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_extraction_frequency=2,
    memory_extraction_window=4,
    enable_context_summary=True,
    context_summary_update_frequency=2,
)
print(f"Smart thread: {smart_thread.thread_id}")


## 14. Watch extraction happen

Add a few messages, then search for `record_type="memory"` — you should find facts the package extracted on its own, without you ever calling `add_memory`.

In [ ]:
smart_thread.add_messages([
    Message(role="user",      content="I prefer Python over TypeScript for backend work."),
    Message(role="assistant", content="Noted. Are you using FastAPI or something else?"),
    Message(role="user",      content="FastAPI, deployed on Oracle Cloud Infrastructure behind an API gateway."),
    Message(role="assistant", content="Makes sense for your scale. Want me to remember that stack?"),
])

extracted = smart_client.search(
    "python stack preferences",
    user_id=USER_ID,
    record_types=["memory"],
    max_results=5,
)
print(f"Extracted {len(extracted)} memories:")
for r in extracted:
    print(f"  - {r.content}")


## 15. Inspect the running summary

With `enable_context_summary=True`, `get_summary()` now produces an LLM-written compression rather than a plain concatenation.


In [ ]:
summary = smart_thread.get_summary()
if summary:
    print(summary[0].content)
else:
    print("(no summary yet — add more messages)")


## 16. Scope searches across users and agents

In production, your package store will have many users and agents sharing one database. `search()` takes scoping arguments so memories never leak across boundaries:

- `user_id=...` — restrict to memories that belong to this user
- `agent_id=...` — restrict to memories created by this agent
- `thread_id=...` — restrict to memories from a specific conversation
- `record_types=[...]` — filter by record kind (`memory`, `message`, `user`, `agent`, `guideline`, `fact`)
- `max_results=N` — cap the result count

Combine them freely.

In [ ]:
# Only memories, scoped to this user, this agent, this thread
scoped = smart_client.search(
    "deployment",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id=smart_thread.thread_id,
    record_types=["memory"],
    max_results=3,
)
for r in scoped:
    print(f"{r.distance:.3f}  {r.content}")


## 17. Use the AI Agent Package inside an LLM agent loop

The typical pattern: **before** every LLM call, pull a context card. **After** the LLM responds, append the assistant message back to the thread. Everything else — extraction, summarisation, embedding — is automatic.

Below is a minimal agent loop using the raw OpenAI client. The same pattern works with any framework.

In [ ]:
from openai import OpenAI
openai_client = OpenAI()

SYSTEM_PROMPT = "You are a helpful assistant. Use the retrieved memory to stay consistent with the user's preferences and history."


def chat_turn(user_message: str) -> str:
    # 1. Persist the user turn (this triggers extraction in the background)
    smart_thread.add_messages([Message(role="user", content=user_message)])

    # 2. Retrieve a compact context card
    card = smart_thread.get_context_card() or "(no prior memory)"

    # 3. Build a small prompt and call the model
    resp = openai_client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Memory context:\n{card}\n\nUser: {user_message}"},
        ],
    )
    answer = resp.choices[0].message.content or ""

    # 4. Persist the assistant reply so future turns can use it
    smart_thread.add_messages([Message(role="assistant", content=answer)])
    return answer


print(chat_turn("What stack do I use for my backend work again?"))


## 18. Persist and resume across sessions

Everything the package stores lives in Oracle, not in your Python process. To resume a conversation from a new process, a new machine, or a new day, just reconnect and call `get_thread(thread_id)`.

In [ ]:
saved_thread_id = smart_thread.thread_id

# Imagine your program restarts here — different connection, different client.
new_connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
new_client = OracleAgentMemory(
    connection=new_connection,
    embedder="text-embedding-3-small",
    llm=Llm("gpt-5.4"),
    extract_memories=True,
    schema_policy="create_if_necessary",
)

recovered = new_client.get_thread(saved_thread_id)
print(f"Recovered thread: {recovered.thread_id}")
print(f"Messages recovered: {len(recovered.get_messages())}")

# The recovered thread is fully writable
recovered.add_messages([Message(role="user", content="Are my preferences still remembered?")])
print(recovered.get_context_card()[:400])


## 19. Cleanup

Delete the threads and memory you created in this walkthrough, then close both connections. The package never deletes anything on its own — you stay in control.

In [ ]:
client.delete_memory(mem_id)
smart_client.delete_thread(smart_thread.thread_id)
new_client.delete_thread(saved_thread_id) if recovered else None
connection.close()
new_connection.close()
print("Resources cleaned up.")


## Where to go next

- **`oracle_agent_memory_quickstart.ipynb`** — end-to-end scenarios exercising every API surface with verification tests (multi-user isolation, search scoping, message CRUD, compaction, long conversations, cross-session continuity, etc.)
- **`oracle_agent_memory_benchmarks.ipynb`** — head-to-head comparison of an agent backed by the AI Agent Package vs a naive flat-history agent across token consumption, wall-clock latency, and response quality (LLM-as-a-judge)

**Suggested reading order for new projects**
1. This guide — to understand the primitives
2. The quickstart scenarios — to see production patterns
3. The benchmarks — to justify the cost/quality tradeoffs to your team